# multi_15m_strat_002

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (103).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `15m` |
| Code cells | 12 |
| Detected functions | resample_1hr_custom, stochastic_oscillator, find_divergences, calculate_quad_stochastic, plot_stochastic_with_divergence, generate_quad_stochastic_signals, _stoch, _divergences, add_stoch_signals, add_stoch_signals_grouped, process_ticker, backtest_trades |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/5min_data.csv')
data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2022-01-01'
end_date = '2024-12-30'

# Define the tickers you want to filter
selected_tickers = ['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK',
       'BAJAJ-AUTO', 'BAJAJFINSV', 'BAJFINANCE', 'BEL', 'BHARTIARTL',
       'BPCL', 'BRITANNIA', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT',
       'GRASIM', 'HCLTECH', 'HEROMOTOCO', 'HINDALCO', 'HINDUNILVR',
       'ICICIBANK', 'INDUSINDBK', 'INFY', 'ITC', 'JSWSTEEL', 'KOTAKBANK',
       'LT', 'MARUTI', 'MM', 'NESTLEIND', 'NTPC', 'ONGC', 'POWERGRID',
       'RELIANCE', 'SBILIFE', 'SBIN', 'SHRIRAMFIN', 'SUNPHARMA',
       'TATACONSUM', 'TATAMOTORS', 'TATASTEEL', 'TCS', 'TECHM', 'TITAN',
       'TRENT', 'ULTRACEMCO', 'WIPRO']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd
'''
# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h', offset='15min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data'''

In [ ]:
data.dropna()
data

In [ ]:
data.columns

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
from plotly.subplots import make_subplots
'''
# --- CONFIGURABLE PARAMETERS ---
stoch_settings = {
    "stoch1": {"k": 9, "d": 3, "smooth_k": 1},
    "stoch2": {"k": 14, "d": 3, "smooth_k": 1},
    "stoch3": {"k": 40, "d": 4, "smooth_k": 1},
    "stoch4": {"k": 60, "d": 10, "smooth_k": 1}
}
left_bars = 5
right_bars = 5

# --- HELPER FUNCTIONS ---
def stochastic_oscillator(data, k_period, d_period, smooth_k=1):
    low_min = data['low'].rolling(window=k_period).min()
    high_max = data['high'].rolling(window=k_period).max()
    k_raw = 100 * (data['close'] - low_min) / (high_max - low_min)
    k_smooth = k_raw.rolling(window=smooth_k).mean()
    d_line = k_smooth.rolling(window=d_period).mean()
    return k_smooth, d_line

def find_divergences(series, price, left=5, right=5):
    bullish_divs = []
    bearish_divs = []

    for i in range(left, len(series) - right):
        is_local_min = all(series[i] < series[i - j] for j in range(1, left + 1)) and \
                       all(series[i] < series[i + j] for j in range(1, right + 1))

        is_local_max = all(series[i] > series[i - j] for j in range(1, left + 1)) and \
                       all(series[i] > series[i + j] for j in range(1, right + 1))

        if is_local_min:
            for j in range(1, 20):
                if i - j < 0: break
                if series[i] > series[i - j] and price[i] < price[i - j]:
                    bullish_divs.append(i)
                    break

        if is_local_max:
            for j in range(1, 20):
                if i - j < 0: break
                if series[i] < series[i - j] and price[i] > price[i - j]:
                    bearish_divs.append(i)
                    break

    return bullish_divs, bearish_divs

# --- MAIN FUNCTION ---
def calculate_quad_stochastic(data):
    data = data.copy()
    for name, params in stoch_settings.items():
        k, d, smooth = params['k'], params['d'], params['smooth_k']
        k_line, d_line = stochastic_oscillator(data, k, d, smooth)
        data[f'{name}_k'] = k_line
        data[f'{name}_d'] = d_line

        # Divergences
        bull_div, bear_div = find_divergences(d_line.values, data['close'].values, left_bars, right_bars)
        data[f'{name}_bullish_div'] = 0
        data[f'{name}_bearish_div'] = 0
        data.iloc[bull_div, data.columns.get_loc(f'{name}_bullish_div')] = 1
        data.iloc[bear_div, data.columns.get_loc(f'{name}_bearish_div')] = 1

    return data

# --- PLOT FUNCTION ---
def plot_stochastic_with_divergence(data):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        vertical_spacing=0.03, row_heights=[0.3, 0.7],
                        subplot_titles=("Stochastic %D lines", "Candlestick + Divergences"))

    # Plot Stochastic %D lines
    for name in stoch_settings:
        fig.add_trace(go.Scatter(x=data['Date'], y=data[f'{name}_d'],
                                 name=f"{name}_d", mode='lines'), row=1, col=1)

        # Bullish markers
        bull_idx = data[data[f'{name}_bullish_div'] == 1]
        fig.add_trace(go.Scatter(x=bull_idx['Date'], y=bull_idx[f'{name}_d'],
                                 mode='markers', name=f'{name} Bull Div',
                                 marker=dict(color='green', symbol='triangle-up', size=10)), row=1, col=1)

        # Bearish markers
        bear_idx = data[data[f'{name}_bearish_div'] == 1]
        fig.add_trace(go.Scatter(x=bear_idx['Date'], y=bear_idx[f'{name}_d'],
                                 mode='markers', name=f'{name} Bear Div',
                                 marker=dict(color='red', symbol='triangle-down', size=10)), row=1, col=1)

    # Candlestick chart
    fig.add_trace(go.Candlestick(x=data['Date'], open=data['open'], high=data['high'],
                                 low=data['low'], close=data['close'], name='Price'), row=2, col=1)

    fig.update_layout(height=800, title="Four Stochastic Oscillators with Divergence Detection",
                      xaxis2_rangeslider_visible=False)
    fig.show()

# --- EXAMPLE USAGE ---

# Load your data (must include: date, open, high, low, close)
# data = pd.read_csv("your_data.csv", parse_dates=['date'])
data = calculate_quad_stochastic(data)
plot_stochastic_with_divergence(data)

def generate_quad_stochastic_signals(data):
    signal = []

    for i in range(1, len(data)):
        bullish_divs = 0
        bearish_divs = 0
        bullish_crosses = 0
        bearish_crosses = 0

        for idx in range(1, 5):
            name = f'Stoch{idx}'
            k_col = f'{name}_%K'
            d_col = f'{name}_%D'
            bull_div_col = f'{name}_bullish_div'
            bear_div_col = f'{name}_bearish_div'

            # Divergence signals
            if data.at[i, bull_div_col] == 1:
                bullish_divs += 1
            if data.at[i, bear_div_col] == 1:
                bearish_divs += 1

            # Crossovers
            if data.at[i - 1, k_col] < data.at[i - 1, d_col] and data.at[i, k_col] > data.at[i, d_col]:
                bullish_crosses += 1
            if data.at[i - 1, k_col] > data.at[i - 1, d_col] and data.at[i, k_col] < data.at[i, d_col]:
                bearish_crosses += 1

        if bullish_divs >= 2 and bullish_crosses >= 2:
            signal.append("buy")
        elif bearish_divs >= 2 and bearish_crosses >= 2:
            signal.append("sell")
        else:
            signal.append("")

    signal.insert(0, "")  # for first row (no signal)
    data["signal"] = signal
    return data

data = calculate_quad_stochastic(data)
data = generate_quad_stochastic_signals(data)
'''

In [ ]:
import pandas as pd
import numpy as np

# ---- Parameters -----------------------------------------------------------
STOCHS = [(9, 3, 1), (14, 3, 1), (40, 4, 1), (60, 10, 1)]  # (k, d, smooth_k)
LEFT = 5  # Only look back

# ---- Helper functions -----------------------------------------------------

def _stoch(data: pd.DataFrame, k: int, d: int, s: int):
    lo = data['low'].rolling(k).min()
    hi = data['high'].rolling(k).max()
    k_raw = 100 * (data['close'] - lo) / (hi - lo)
    k_s = k_raw.rolling(s).mean()
    d_line = k_s.rolling(d).mean()
    return k_s, d_line

def _divergences(osc: np.ndarray, price: np.ndarray):
    bull, bear = [], []
    for i in range(LEFT, len(osc)):
        # Only look at past to detect valleys and peaks
        if (osc[i] < osc[i - LEFT:i]).all():
            for j in range(1, 20):
                if i - j < 0:
                    break
                if osc[i] > osc[i - j] and price[i] < price[i - j]:
                    bull.append(i)
                    break
        if (osc[i] > osc[i - LEFT:i]).all():
            for j in range(1, 20):
                if i - j < 0:
                    break
                if osc[i] < osc[i - j] and price[i] > price[i - j]:
                    bear.append(i)
                    break
    return bull, bear

# ---- Main signal calculation ----------------------------------------------

def add_stoch_signals(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data.sort_values("Date", inplace=True)
    data.reset_index(drop=True, inplace=True)

    bulls_tot = np.zeros(len(data), dtype=int)
    bears_tot = np.zeros(len(data), dtype=int)
    bull_x_tot = np.zeros(len(data), dtype=int)
    bear_x_tot = np.zeros(len(data), dtype=int)

    for k, d, s in STOCHS:
        k_line, d_line = _stoch(data, k, d, s)
        bull_div, bear_div = _divergences(d_line.values, data['close'].values)

        bulls = np.zeros(len(data), dtype=int)
        bulls[bull_div] = 1
        bears = np.zeros(len(data), dtype=int)
        bears[bear_div] = 1

        bulls_tot += bulls
        bears_tot += bears

        # Crossover detection
        k_shift = k_line.shift(1)
        d_shift = d_line.shift(1)
        bull_x = ((k_shift < d_shift) & (k_line > d_line)).astype(int)
        bear_x = ((k_shift > d_shift) & (k_line < d_line)).astype(int)

        bull_x_tot += bull_x.fillna(0).astype(int)
        bear_x_tot += bear_x.fillna(0).astype(int)

    signal = np.where(
        (bulls_tot >= 2) & (bull_x_tot >= 2), 1,
        np.where((bears_tot >= 2) & (bear_x_tot >= 2), 2, 0)
    )
    data['Signal'] = signal
    return data

# ---- Group by ticker wrapper ----------------------------------------------

def add_stoch_signals_grouped(data: pd.DataFrame) -> pd.DataFrame:
    return data.groupby('Ticker', group_keys=False).apply(add_stoch_signals).reset_index(drop=True)

# ---- Example usage --------------------------------------------------------

# # Example:
# data = pd.read_csv("your_data.csv")  # Your data should have columns: Ticker, date, open, high, low, close, volume
data= add_stoch_signals_grouped(data)
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from joblib import Parallel, delayed

# Function to process trades for a single ticker
def process_ticker(ticker, ticker_data, initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time):
    """Processes trades for a single ticker, applying entry/exit logic and calculating PnL."""
    open_positions = {}
    trades = []
    cumulative_pnl = 0

    for row in ticker_data:
        timestamp, close, open_price, high, low, signal = row["date"], row["close"], row["open"], row["high"], row["low"], row["Signal"]

        # Check for open positions and apply exit logic
        if ticker in open_positions:
            position = open_positions[ticker]
            entry_price, direction, qty = position["entry_price"], position["direction"], position["qty"]
            exit_price, exit_reason = None, None

            # Compute Take Profit (TP) & Stop Loss (SL) prices
            tp_price = entry_price * (1 + tp_pct) if direction == "long" else entry_price * (1 - tp_pct)
            sl_price = entry_price * (1 - sl_pct) if direction == "long" else entry_price * (1 + sl_pct)

            # Determine exit conditions
            if (direction == "long" and high >= tp_price) or (direction == "short" and low <= tp_price):
                exit_price, exit_reason = tp_price, "Take Profit"
            elif (direction == "long" and low <= sl_price) or (direction == "short" and high >= sl_price):
                exit_price, exit_reason = sl_price, "Stop Loss"
            elif timestamp.astype('datetime64[m]').astype(str)[-5:] >= end_time:
                exit_price, exit_reason = close, "EOD Close"

            if exit_price is not None:
                # Apply slippage
                exit_price *= (1 - slippage) if direction == "long" else (1 + slippage)

                # Calculate PnL including brokerage
                raw_pnl = (exit_price - entry_price) * qty if direction == "long" else (entry_price - exit_price) * qty
                pnl = raw_pnl - (brokerage * initial_capital * 2)
                cumulative_pnl += pnl

                # Store trade details
                trades.append([ticker, position["entry_time"], timestamp, entry_price, exit_price, pnl, exit_reason, cumulative_pnl, direction])
                del open_positions[ticker]
                continue

        # Open new position
        if ticker not in open_positions and timestamp.astype('datetime64[m]').astype(str)[-5:] < stop_trading_time:
            if signal == 1:
                direction = "long"
            elif signal == 2:
                direction = "short"
            else:
                continue

            entry_price = open_price * (1 + slippage) if direction == "long" else open_price * (1 - slippage)
            qty = initial_capital / entry_price
            open_positions[ticker] = {"entry_time": timestamp, "entry_price": entry_price, "direction": direction, "qty": qty}

    return trades

# Main backtesting function
def backtest_trades(data):
    """Runs backtest for all tickers in the dataset using parallel processing."""

    # Trading parameters
    initial_capital = 100000  # ₹1 lakh per trade
    brokerage = 0.001  # 0.1% brokerage
    slippage = 0.000  # 0.01% slippage
    tp_pct = 0.03  # 1.5% take-profit
    sl_pct = 0.005  # 0.5% stop-loss
    end_time = "15:15"
    stop_trading_time = "14:15"

    # Get unique tickers
    unique_tickers = data['Ticker'].unique()

    # Convert DataFrame to NumPy structured array for performance
    dtype = [
        ("Ticker", "U10"), ("date", "M8[ms]"), ("close", "f8"),
        ("open", "f8"), ("high", "f8"), ("low", "f8"), ("Signal", "i4")
    ]

    data_np = np.array(
        list(data[["Ticker", "Date", "close", "open", "high", "low", "Signal"]]
            .itertuples(index=False, name=None)),
        dtype=dtype
    )

    # Run backtest in parallel
    results = Parallel(n_jobs=-1)(delayed(process_ticker)(ticker, data_np[data_np['Ticker'] == ticker], initial_capital, brokerage, slippage, tp_pct, sl_pct, end_time, stop_trading_time) for ticker in tqdm(unique_tickers))

    # Flatten results into DataFrame
    trades = [trade for result in results for trade in result]
    return pd.DataFrame(trades, columns=["Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price", "PnL", "Exit Reason", "Cumulative PnL", "Direction"])

# Example usage
# data = pd.read_csv("your_data.csv")  # Load dataset
data['Date'] = pd.to_datetime(data['Date'])  # Ensure Date column is in datetime format
tradebook = backtest_trades(data)
tradebook.to_csv("/content/drive/MyDrive/tradebook.csv", index=False)
tradebook

In [ ]:
from itertools import product
import numpy as np
import pandas as pd
from tqdm import tqdm

# Parameter grids (no stoch hypertuning)
left_list = [3, 4, 5, 6, 7]
right_list = [4, 5, 6, 7]
tp_list = [0.03]
sl_list = [0.01]

# Prepare full parameter combinations
param_combinations = list(product(left_list, right_list, tp_list, sl_list))

# Store results
results = []

# Grid search with tqdm
for counter, (left, right, tp, sl) in enumerate(tqdm(param_combinations, desc="Grid Search"), 1):

    LEFT = left
    RIGHT = right
    TP = tp
    SL = sl

    try:
        # Generate signals (using predefined STOCHS internally)
        data_with_signals = add_stoch_signals_grouped(data)

        # Backtest
        tradebook = backtest_trades(data_with_signals)

        if tradebook.empty:
            continue

        total_profit = tradebook["PnL"].sum()
        cumulative_pnl = tradebook["Cumulative PnL"]
        drawdown = cumulative_pnl.cummax() - cumulative_pnl
        max_drawdown = drawdown.max()
        profit_dd_ratio = total_profit / max_drawdown if max_drawdown != 0 else np.nan

        result = {
            "Run": counter,
            "LEFT": left,
            "RIGHT": right,
            "TP": tp,
            "SL": sl,
            "Total Profit": round(total_profit, 2),
            "Max Drawdown": round(max_drawdown, 2),
            "Profit/DD": round(profit_dd_ratio, 3),
            "Trades": len(tradebook)
        }

        results.append(result)

        # Print result
        print(f"\nRun {counter}:")
        print(f"  LEFT = {left}, RIGHT = {right}")
        print(f"  TP = {tp*100:.2f}%, SL = {sl*100:.2f}%")
        print(f"  → Total Profit = ₹{total_profit:.2f}")
        print(f"  → Max Drawdown = ₹{max_drawdown:.2f}")
        print(f"  → Profit/DD Ratio = {profit_dd_ratio:.3f}")
        print(f"  → Total Trades = {len(tradebook)}")
        print("-" * 50)

    except Exception as e:
        print(f"\nRun {counter} failed: {e}")
        continue

# Results DataFrame
results_df = pd.DataFrame(results)
